In [1]:
import logging
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, SummarizationMiddleware
from langchain_core.runnables import RunnableConfig

In [2]:
from pydantic import BaseModel, ConfigDict, Field


class ModelConfig(BaseModel):
    """Config section for a model"""

    # 唯一标识名称
    name: str = Field(..., description="Unique name for the model")
    # 模型官方名称
    model: str = Field(..., description="Model name")
    # API 基础地址
    base_url: str = Field(..., description="Base URL for the model API")
    # API 密钥
    api_key: str = Field(..., description="API key for authentication")

    # 可选描述信息
    description: str | None = Field(
        default=None,
        description="Description for the model"
    )
    # 请求超时时间（秒）
    timeout: float = Field(
        default=60.0,
        description="Request timeout in seconds"
    )
    # 最大重试次数
    max_retries: int = Field(
        default=2,
        description="Maximum number of retries for failed requests"
    )
    # 是否支持思考过程
    supports_thinking: bool = Field(
        default=False,
        description="Whether the model supports thinking/chain-of-thought"
    )
    # 是否支持视觉能力
    supports_vision: bool = Field(
        default=False,
        description="Whether the model supports vision/image inputs"
    )
    # 是否支持推理强度配置
    supports_reasoning_effort: bool = Field(
        default=False,
        description="Whether the model supports reasoning effort adjustment"
    )

    # 允许额外字段
    model_config = ConfigDict(extra="allow")

In [3]:
# model = ModelConfig(
#     name="gpt-4o",
#     model="gpt-4o",
#     api_base="https://api.openai.com/v1",
#     api_key="your-api-key"
# )

In [4]:
import logging
import os
from contextvars import ContextVar
from pathlib import Path
from typing import Any, Self

import yaml
from dotenv import load_dotenv
from pydantic import BaseModel, ConfigDict, Field

# from .model_config import ModelConfig

load_dotenv()

logger = logging.getLogger(__name__)


class AppConfig(BaseModel):
    """Config for the optclaw application"""

    log_level: str = Field(default="info", description="Logging level for optclaw modules (debug/info/warning/error)")
    models: list[ModelConfig] = Field(default_factory=list, description="Available models")


    @classmethod
    def from_file(cls, config_path: str | None = None) -> Self:
        """Load config from YAML file.

        Args:
            config_path: Path to the config file.

        Returns:
            AppConfig: The loaded config.
        """
        with open(config_path, encoding="utf-8") as f:
            config_data = yaml.safe_load(f) or {}

        return cls.model_validate(config_data)

    def get_model_config(self, name: str) -> ModelConfig | None:
        """Get the model config by name.

        Args:
            name: The name of the model to get the config for.

        Returns:
            The model config if found, otherwise None.
        """
        print(f"Looking for model config with name: {name}")
        return next((model for model in self.models if model.name == name), None)


_app_config: AppConfig | None = None


def _get_config_mtime(config_path: Path) -> float | None:
    """Get the modification time of a config file if it exists."""
    try:
        return config_path.stat().st_mtime
    except OSError:
        return None


def get_app_config() -> AppConfig:
    """Get the config instance.
    """
    global _app_config

    # read yaml
    config_path = Path('../../config.yaml')
    current_mtime = _get_config_mtime(config_path)

    _app_config = AppConfig.from_file(str(config_path))

    logger.info(f"Loading config from {config_path} (mtime={current_mtime})")

    return _app_config

In [5]:
import logging
from langchain.chat_models import BaseChatModel
from langchain_openai import ChatOpenAI

from config import get_app_config


logger = logging.getLogger(__name__)


def create_chat_model(name: str | None = None, thinking_enabled: bool = False, **kwargs) -> ChatOpenAI:
    """Create a chat model instance from the config.

    Args:
        name: The name of the model to create. If None, the first model in the config will be used.

    Returns:
        A chat model instance.
    """
    config = get_app_config()
    if name is None:
        name = config.models[0].name
    model_config = config.get_model_config(name)
    if model_config is None:
        raise ValueError(f"Model {name} not found in config") from None
    
    model_config = model_config.model_dump(
    exclude_none=True,
    exclude={
        "use",
        "name",
        "display_name",
        "description",
        "supports_thinking",
        "supports_reasoning_effort",
        "when_thinking_enabled",
        "thinking",
        "supports_vision",
    },
    )
    
    model:ChatOpenAI = ChatOpenAI(**model_config)

    return model


In [ ]:
model = create_chat_model()

In [ ]:
# 创建Agent
agent = create_agent(
    model=model,
    system_prompt="你好"
)

# 正确调用
response = agent.invoke({
    "messages": [{"role": "user", "content": "今天天气怎么样？"}]
})

print(response)

{'messages': [HumanMessage(content='今天天气怎么样？', additional_kwargs={}, response_metadata={}, id='7286a93d-8927-40da-b7a9-fd3549c628a4'), AIMessage(content='我没办法实时获取当前的天气信息呢😅 你可以通过手机自带的天气APP、微信/支付宝的天气小程序，或者像中央气象台官网这样的平台，输入你所在的城市，就能查到精准的天气情况啦～', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 121, 'prompt_tokens': 42, 'total_tokens': 163, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 70, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'doubao-seed-1-8-251228', 'system_fingerprint': None, 'id': '02177699597187277b7e892e9721221f8ca1a94f85371e51f7495', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dbd36-b2cf-7411-9ace-f1b6b8ab0dd9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 121, 'total_tokens': 163,

In [1]:
from optclaw.config import get_app_config
from optclaw.models import create_chat_model
from langchain.agents import create_agent

In [2]:
model = create_chat_model(thinking_enabled=True)

Creating model 'doubao-seed-1.8' with settings: {'model': 'doubao-seed-1-8-251228', 'base_url': 'https://ark.cn-beijing.volces.com/api/v3', 'api_key': '31d61644-628b-4332-9e5c-1987c99f3427', 'timeout': 600.0, 'max_retries': 3, 'extra_body': {'thinking': {'type': 'enabled'}}} and kwargs: {}


In [3]:
# 创建Agent
agent = create_agent(
    model=model,
    system_prompt="你好"
)

# 正确调用
response = agent.invoke({
    "messages": [{"role": "user", "content": "今天天气怎么样？"}]
})

print(response)

{'messages': [HumanMessage(content='今天天气怎么样？', additional_kwargs={}, response_metadata={}, id='9db0729e-622a-43ae-92da-f073a9ae9340'), AIMessage(content='我没办法实时获取当前的天气信息哦😉。你可以通过手机自带的天气APP、百度搜索“当地天气”，或者查看天气预报网站，就能查到准确的实时天气啦！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 124, 'prompt_tokens': 42, 'total_tokens': 166, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 83, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'doubao-seed-1-8-251228', 'system_fingerprint': None, 'id': '0217770455910577fbfe5bd10ef69e0c508f864fad73fe0a13fd6', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dc02b-d55d-7563-9bea-f0cc66fb593e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 124, 'total_tokens': 166, 'input_token_de

In [ ]:
# from optclaw.config.paths import Paths

ModuleNotFoundError: No module named 'optclaw'

In [ ]:
# Path(f'{Paths().base_dir}/config.yaml')

In [3]:
Path(__file__).resolve().parents[4]

NameError: name '__file__' is not defined

In [ ]:
# Agent 的实例化不是单一位置，而是一个分层架构。让我为你梳理完整的实例化链条。

# ## Agent 实例化的完整路径

# ### 1. 最底层：LangChain 原语
# `langchain.agents.create_agent` 是真正的底层创建函数，它返回一个 `CompiledStateGraph`（编译后的状态图）。DeerFlow 的所有工厂都最终调用它。

# ### 2. SDK 层：纯参数工厂
# [create_deerflow_agent](backend/packages/harness/deerflow/agents/factory.py#L61-L154)  
# 这是 SDK 级别的入口点。它接受纯 Python 参数（模型、工具、中间件、系统提示等），不读取任何配置文件。如果你想在代码中直接创建一个 agent 而不依赖配置系统，就用它。

# ```python
# # SDK 用法示例
# agent = create_deerflow_agent(
#     model=ChatOpenAI(model="gpt-4"),
#     tools=[my_tool],
#     system_prompt="You are a helpful assistant",
# )
# ```

# ### 3. 应用层：配置驱动工厂
# [make_lead_agent](backend/packages/harness/deerflow/agents/lead_agent/agent.py#L280-L358)  
# 这是生产环境的入口。它从配置文件读取设置（模型名称、thinking 模式、是否启用子 agent、工具组等），然后调用 `create_agent`。整个系统启动时，实际走的是这条路径。

# ### 4. 运行时：执行入口
# [backend/packages/harness/deerflow/runtime/runs/](backend/packages/harness/deerflow/runtime/runs/) 中的 `manager.py` 和 `worker.py` 负责实际的 agent 实例调用和执行流管理。这里不是创建 agent，而是把已经创建好的 agent 图实例化运行。

# ## 总结
# - **如果你在开发 SDK** 或写单元测试：直接用 [create_deerflow_agent](backend/packages/harness/deerflow/agents/factory.py#L61-L154)  
# - **如果你在部署生产环境**：实例化通过 [make_lead_agent](backend/packages/harness/deerflow/agents/lead_agent/agent.py#L280-L358) 完成，由配置驱动  
# - **底层真相**：最终都是 LangChain 的 `create_agent` 函数创建状态图

# 如果你需要了解运行时如何调度这些 agent，可以查看 [运行时管理](backend/packages/harness/deerflow/runtime/runs/manager.py)。

# [主导 Agent 构建](10-lead-agent-construction)  
# [中间件系统设计](12-middleware-system-design)  
# [架构概述](8-architecture-overview)

In [ ]:
# #!/usr/bin/env python
# """
# Debug script for lead_agent.
# Run this file directly in VS Code with breakpoints.

# Requirements:
#     Run with `uv run` from the backend/ directory so that the uv workspace
#     resolves deerflow-harness and app packages correctly:

#         cd backend && PYTHONPATH=. uv run python debug.py

# Usage:
#     1. Set breakpoints in agent.py or other files
#     2. Press F5 or use "Run and Debug" panel
#     3. Input messages in the terminal to interact with the agent
# """

# import asyncio
# import logging

# from dotenv import load_dotenv
# from langchain_core.messages import HumanMessage

# from deerflow.agents import make_lead_agent

# load_dotenv()

# logging.basicConfig(
#     level=logging.INFO,
#     format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
#     datefmt="%Y-%m-%d %H:%M:%S",
# )


# async def main():
#     # Initialize MCP tools at startup
#     try:
#         from deerflow.mcp import initialize_mcp_tools

#         await initialize_mcp_tools()
#     except Exception as e:
#         print(f"Warning: Failed to initialize MCP tools: {e}")

#     # Create agent with default config
#     config = {
#         "configurable": {
#             "thread_id": "debug-thread-001",
#             "thinking_enabled": True,
#             "is_plan_mode": True,
#             # Uncomment to use a specific model
#             "model_name": "kimi-k2.5",
#         }
#     }

#     agent = make_lead_agent(config)

#     print("=" * 50)
#     print("Lead Agent Debug Mode")
#     print("Type 'quit' or 'exit' to stop")
#     print("=" * 50)

#     while True:
#         try:
#             user_input = input("\nYou: ").strip()
#             if not user_input:
#                 continue
#             if user_input.lower() in ("quit", "exit"):
#                 print("Goodbye!")
#                 break

#             # Invoke the agent
#             state = {"messages": [HumanMessage(content=user_input)]}
#             result = await agent.ainvoke(state, config=config, context={"thread_id": "debug-thread-001"})

#             # Print the response
#             if result.get("messages"):
#                 last_message = result["messages"][-1]
#                 print(f"\nAgent: {last_message.content}")

#         except KeyboardInterrupt:
#             print("\nInterrupted. Goodbye!")
#             break
#         except Exception as e:
#             print(f"\nError: {e}")
#             import traceback

#             traceback.print_exc()


# if __name__ == "__main__":
#     asyncio.run(main())


In [6]:
from typing import TypedDict

class User(TypedDict):
    name: str
    age: int